In [ ]:
# Feature Importance Analysis (for tree-based models)
print("\n" + "=" * 70)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

for name in ['Random Forest', 'Gradient Boosting']:
    if name in results:
        model = results[name]['model']
        feature_importance = pd.DataFrame({
            'Feature': selected_features,
            'Importance': model.feature_importances_
        }).sort_values('Importance', ascending=False)
        
        print(f"\n{name} - Top 10 Most Important Features:")
        print(feature_importance.head(10).to_string(index=False))
        
        # Calculate cumulative importance
        cum_importance = feature_importance['Importance'].cumsum()
        num_features_90 = (cum_importance <= 0.9).sum() + 1
        print(f"\nNumber of features explaining 90% of variance: {num_features_90}")


## Step 4: Model Selection Documentation

### Selected Baseline Model

Based on the comparison above, we recommend using **Random Forest** or **Gradient Boosting** as the baseline model depending on your priorities:

- **Random Forest**: Better for interpretability and faster training
- **Gradient Boosting**: Better for maximum accuracy

### Why This Model is Appropriate

1. **Multiple External Factors**: Our dataset includes temperature, fuel price, CPI, unemployment, and markdowns that influence sales
2. **Non-linear Relationships**: Sales don't follow simple linear patterns with these economic indicators
3. **Feature Importance**: Tree ensembles provide feature importance for business insights
4. **Categorical Variables**: Store and Department are naturally handled by tree-based models
5. **Robustness**: Ensemble methods are resistant to overfitting and outliers
6. **Scalability**: Can handle 421K+ training records efficiently

### Assumptions & Limitations

1. **Stationarity**: Assumes sales patterns remain relatively stable over time
2. **Feature Relevance**: Assumes provided features are relevant predictors
3. **External Changes**: May not capture sudden market shifts or new events
4. **Data Quality**: Performance depends on quality of feature data (holidays, markdowns)
5. **Store Diversity**: Performance may vary across different store types
6. **Seasonality**: Model is not explicitly designed for seasonal patterns (rely on lag features)

## Step 3: Model Strengths & Weaknesses

| Model | Strengths | Weaknesses | Use Case |
|-------|-----------|-----------|----------|
| Linear Regression | Fast, interpretable, low memory | Assumes linear relationships | Quick baseline |
| Decision Tree | Handles non-linearity, easy visualization | Prone to overfitting | Shallow/moderate complexity |
| Random Forest | Robust, good generalization, feature importance | Slower training, less interpretable | Complex patterns with many features |
| Gradient Boosting | High accuracy, handles complex relationships | Slower, more hyperparameters | When accuracy is critical |

### When to Use Which Model

- **Linear Regression**: Use as a baseline to establish minimum performance
- **Tree-based**: Use when you need to capture non-linear patterns  
- **Ensemble Methods**: Use when you have sufficient data and need high accuracy
- **Time-Series Models**: Use only if forecast is based primarily on historical sales alone

In [ ]:
# Define and train models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, max_depth=4, learning_rate=0.1, random_state=42)
}

results = {}

print("=" * 70)
print("MODEL TRAINING AND EVALUATION")
print("=" * 70)

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    
    # Make predictions
    train_preds = model.predict(X_train)
    valid_preds = model.predict(X_valid)
    
    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    valid_mae = mean_absolute_error(y_valid, valid_preds)
    valid_r2 = r2_score(y_valid, valid_preds)
    
    results[name] = {
        'train_rmse': train_rmse,
        'valid_rmse': valid_rmse,
        'valid_mae': valid_mae,
        'valid_r2': valid_r2,
        'model': model
    }
    
    print(f"  Train RMSE: {train_rmse:.2f}")
    print(f"  Valid RMSE: {valid_rmse:.2f}")
    print(f"  Valid MAE:  {valid_mae:.2f}")
    print(f"  Valid R²:   {valid_r2:.4f}")

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Train RMSE': [results[m]['train_rmse'] for m in results],
    'Valid RMSE': [results[m]['valid_rmse'] for m in results],
    'Valid MAE': [results[m]['valid_mae'] for m in results],
    'Valid R²': [results[m]['valid_r2'] for m in results]
}, index=results.keys())

print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
print(comparison_df.round(2))

# Rank models by validation RMSE
ranked = sorted(results.items(), key=lambda x: x[1]['valid_rmse'])
print("\nMODEL RANKINGS (by Validation RMSE):")
for rank, (name, metrics) in enumerate(ranked, 1):
    print(f"{rank}. {name:20s} - RMSE: {metrics['valid_rmse']:.2f}, R²: {metrics['valid_r2']:.4f}")

best_name = ranked[0][0]
best_model = ranked[0][1]['model']
print(f"\n✓ Best Model: {best_name}")


## Step 2: Compare Regression-Based Models

We will train and evaluate the following models:
1. Linear Regression (simple baseline)
2. Decision Tree (non-linear patterns)
3. Random Forest (ensemble approach)
4. Gradient Boosting (advanced ensemble)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Import our sales forecasting functions
from salesforecasting import (load_data, clean_data, merge_datasets, 
                             engineer_features, select_features)

# Load and prepare data
train, test, stores, features = load_data('.')
train, test, stores, features = clean_data(train, test, stores, features)
train, test = merge_datasets(train, test, stores, features)

# Feature engineering
X, y, feature_columns = engineer_features(train)
selected_features = select_features(X, y, feature_columns, method='correlation')
X = X[selected_features]

# Train-validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dataset prepared:")
print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_valid.shape}")
print(f"Features: {len(selected_features)}")


## Step 1: Identify Suitable Machine Learning Models

Sales forecasting can be addressed using two main categories of models:

### 1. Regression-Based Models
These models handle structured data with multiple features and external factors:
- **Linear Regression**: Simple, fast, interpretable baseline
- **Decision Trees**: Captures non-linear patterns, easy to understand
- **Random Forests**: Ensemble of trees, robust and handles complex relationships
- **Gradient Boosting (XGBoost, LightGBM)**: High accuracy with feature importance

### 2. Time-Series Models
These models specialize in temporal dependencies and sequential patterns:
- **ARIMA**: Traditional approach, requires stationary data
- **SARIMA**: Handles seasonality in time series
- **LSTMs**: Deep learning for complex sequential patterns

### Why Regression Models for This Task
For our Walmart sales data with store, department, holidays, and economic indicators, **regression-based models** are more appropriate because:
- We have rich external features (temperature, fuel price, markdowns, CPI, unemployment)
- Data is structured with multiple predictors affecting sales
- Store and department are categorical features that need handling
- Holiday and promotional effects are key factors

# Selecting Machine Learning Models for Sales Forecasting

This notebook demonstrates the process of selecting the right machine learning models for sales forecasting. We will identify suitable models, compare their performance, and select a baseline model for evaluation.